# Model Training & Evaluation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_ratings.csv")
print(f"Loaded: {len(df):,} ratings | "
      f"{df['book_id'].nunique():,} books | "
      f"{df['user_id'].nunique():,} users")

Loaded: 4,044,839 ratings | 22,931 books | 61,078 users


## 1. Train/Validation/Test Split

In [2]:
# Standard 80/10/10 train/val/test split
n = len(df)
test_size = int(n * 0.10)
val_size  = int(n * 0.10)

test  = df.sample(n=test_size, random_state=42)
rest  = df.drop(test.index)
val   = rest.sample(n=val_size, random_state=42)
train = rest.drop(val.index)

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f"Train: {len(train):,}  ({len(train)/n*100:.1f}%)")
print(f"Val:   {len(val):,}   ({len(val)/n*100:.1f}%)")
print(f"Test:  {len(test):,}   ({len(test)/n*100:.1f}%)")

# Cold-start check
for name, split in [("Val", val), ("Test", test)]:
    cold_u = set(split["user_id"]) - set(train["user_id"])
    cold_b = set(split["book_id"]) - set(train["book_id"])
    print(f"{name} cold-start — users: {len(cold_u)}, books: {len(cold_b)}")

train.to_csv("train_ratings.csv", index=False)
val.to_csv("val_ratings.csv",     index=False)
test.to_csv("test_ratings.csv",   index=False)
print("\nSaved: train_ratings.csv, val_ratings.csv, test_ratings.csv")

Train: 3,235,873  (80.0%)
Val:   404,483   (10.0%)
Test:  404,483   (10.0%)


Val cold-start — users: 0, books: 0


Test cold-start — users: 0, books: 0



Saved: train_ratings.csv, val_ratings.csv, test_ratings.csv


## 2. Train/Val/Test Validation

In [3]:
train = pd.read_csv("train_ratings.csv")
val   = pd.read_csv("val_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

ORIGINAL_TOTAL = 4_044_839
n = len(train) + len(val) + len(test)

print(f"{'CHECK':<45} RESULT")
print("─" * 68)

# 1. Total count preserved
status = "✓" if n == ORIGINAL_TOTAL else f"⚠ expected {ORIGINAL_TOTAL:,}"
print(f"{'1. Total count':<45} {n:,}  {status}")

# 2. Proportions (~80 / 10 / 10)
print(f"{'2. Proportions (train/val/test)':<45} "
      f"{len(train)/n*100:.1f}% / {len(val)/n*100:.1f}% / {len(test)/n*100:.1f}%")

# 3. No nulls
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    nulls = df.isnull().sum().sum()
    print(f"{'3. Nulls (' + name + ')':<45} {'✓ None' if nulls == 0 else f'⚠ {nulls}'}")

# 4. Rating range [1–5]
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    rmin, rmax = df["rating"].min(), df["rating"].max()
    ok = rmin >= 1 and rmax <= 5
    print(f"{'4. Rating range (' + name + ')':<45} {'✓' if ok else '⚠'} [{rmin}, {rmax}]")

# 5. No cross-split overlap
# Index-based splitting + total count preservation guarantees disjoint splits
# (same (user_id, book_id) pair cannot appear in two splits since original has no duplicates)
print(f"{'5. No cross-split overlap':<45} "
      f"{'✓ guaranteed (index-based split + total preserved)' if n == ORIGINAL_TOTAL else '⚠'}")

# 6. Cold-start check
cold_val_u  = len(set(val["user_id"])  - set(train["user_id"]))
cold_test_u = len(set(test["user_id"]) - set(train["user_id"]))
cold_val_b  = len(set(val["book_id"])  - set(train["book_id"]))
cold_test_b = len(set(test["book_id"]) - set(train["book_id"]))
print(f"{'6a. Cold-start users (val / test)':<45} {cold_val_u} / {cold_test_u}")
print(f"{'6b. Cold-start books (val / test)':<45} {cold_val_b} / {cold_test_b}")

# 7. Rating distribution (should be similar across splits)
print(f"\n{'7. Mean rating':<45} "
      f"Train: {train['rating'].mean():.3f} | "
      f"Val: {val['rating'].mean():.3f} | "
      f"Test: {test['rating'].mean():.3f}")
print(f"{'   Std rating':<45} "
      f"Train: {train['rating'].std():.3f}  | "
      f"Val: {val['rating'].std():.3f}  | "
      f"Test: {test['rating'].std():.3f}")

CHECK                                         RESULT
────────────────────────────────────────────────────────────────────
1. Total count                                4,044,839  ✓
2. Proportions (train/val/test)               80.0% / 10.0% / 10.0%
3. Nulls (Train)                              ✓ None
3. Nulls (Val)                                ✓ None
3. Nulls (Test)                               ✓ None
4. Rating range (Train)                       ✓ [1, 5]
4. Rating range (Val)                         ✓ [1, 5]
4. Rating range (Test)                        ✓ [1, 5]
5. No cross-split overlap                     ✓ guaranteed (index-based split + total preserved)


6a. Cold-start users (val / test)             0 / 0
6b. Cold-start books (val / test)             0 / 0

7. Mean rating                                Train: 3.988 | Val: 3.986 | Test: 3.987
   Std rating                                 Train: 0.937  | Val: 0.937  | Test: 0.939


## 3. Surprise SVD — Baseline

Train a default-parameter SVD on the full training set as a reference point before tuning.
SVD (matrix factorization) represents each user and book as a latent vector of length
`n_factors`; the predicted rating is their dot product plus global/user/book bias terms,
which absorb the dataset's strong positive skew (mean rating ≈ 3.99).

In [4]:
from surprise import SVD, Dataset, Reader, accuracy

reader = Reader(rating_scale=(1, 5))
train = pd.read_csv("train_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

# Surprise trainset (full train) + test set as (user, book, rating) tuples
trainset = Dataset.load_from_df(train[["user_id", "book_id", "rating"]], reader).build_full_trainset()
testset  = list(test[["user_id", "book_id", "rating"]].itertuples(index=False, name=None))

baseline = SVD(random_state=42)  # defaults: n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02
baseline.fit(trainset)

rmse_baseline = accuracy.rmse(baseline.test(testset), verbose=False)
print(f"Baseline SVD (defaults) — test RMSE: {rmse_baseline:.4f}")

Baseline SVD (defaults) — test RMSE: 0.7614


## 4. Hyperparameter Tuning (GridSearchCV)

`GridSearchCV` runs 3-fold cross-validation over the parameter grid on the full training
set and keeps the combination with the lowest CV RMSE. `n_factors` is the number of
latent dimensions used to describe each user/book; `n_epochs`, `lr_all`, and `reg_all`
control training length, learning rate, and regularization strength.

In [5]:
import json
from surprise.model_selection import GridSearchCV

full_ds = Dataset.load_from_df(train[["user_id", "book_id", "rating"]], reader)

# Grid centered on the defaults (lr_all=0.005, reg_all=0.02) and extended outward
# (n_factors up to 200) so the optimum is bracketed inside the grid, not on a boundary.
param_grid = {
    "n_factors": [100, 150, 200],
    "n_epochs":  [20, 30, 40],
    "lr_all":    [0.005, 0.01, 0.02],
    "reg_all":   [0.02, 0.05, 0.1],
}

gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1)
gs.fit(full_ds)  # 81 x 3 = 243 fits on the full 3.2M-row train set (~30 min)

best_params = gs.best_params["rmse"]
print(f"Best CV RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best params:  {best_params}")
json.dump(best_params, open("best_params.json", "w"))

Best CV RMSE: 0.7574
Best params:  {'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.02, 'reg_all': 0.1}


## 5. Final Model & Evaluation

Refit SVD on the full training set with the tuned parameters and report RMSE on the
held-out test set, next to the baseline. On this large, sparse dataset most of the
predictable variance is already captured by the bias terms, so tuning moves test RMSE
only marginally — the two models are effectively tied. The fitted model is saved
(`svd_model.pkl`) for the console recommender.

In [6]:
import pickle

best_params = json.load(open("best_params.json"))
tuned = SVD(**best_params, random_state=42)
tuned.fit(trainset)
rmse_tuned = accuracy.rmse(tuned.test(testset), verbose=False)

print(f"Baseline test RMSE: {rmse_baseline:.4f}")
print(f"Tuned    test RMSE: {rmse_tuned:.4f}   (params: {best_params})")

# Save the tuned model for the console recommender (recommend.py)
with open("svd_model.pkl", "wb") as f:
    pickle.dump(tuned, f)
print("Saved final model -> svd_model.pkl")

Baseline test RMSE: 0.7614
Tuned    test RMSE: 0.7627   (params: {'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.02, 'reg_all': 0.1})


Saved final model -> svd_model.pkl


## 6. Book Title Mapping

The ratings table only stores `book_id`. Build a `book_id → title` lookup from the
Goodreads metadata file in a single pass (restricted to the books that survived
cleaning) so the console recommender can show real titles.

In [7]:
import gzip, json

# Restrict the lookup to the books that survived cleaning (one pass over metadata)
cleaned_books = set(pd.read_csv("cleaned_ratings.csv", usecols=["book_id"])["book_id"].astype(str))

titles = {}
with gzip.open("goodreads_books_children.json.gz", "rt") as f:
    for line in f:
        rec = json.loads(line)
        bid = rec.get("book_id")
        if bid in cleaned_books:
            titles[bid] = (rec.get("title_without_series") or rec.get("title") or "").strip()

title_df = pd.DataFrame(sorted(titles.items()), columns=["book_id", "title"])
title_df.to_csv("book_titles.csv", index=False)
print(f"Saved {len(title_df):,} titles -> book_titles.csv "
      f"({len(titles) / len(cleaned_books) * 100:.1f}% of {len(cleaned_books):,} cleaned books covered)")

Saved 22,931 titles -> book_titles.csv (100.0% of 22,931 cleaned books covered)
